# NiyamTrace-X Standalone 3 — Transaction-bound broker, TOCTOU, replay and latency

Upload this notebook alone to Colab and run top-to-bottom. It is CPU-only and contains the complete SQLite/HMAC experiment.

## Scope

This notebook runs the full **1,320-scenario broker matrix** (120 randomized states × 11 conditions), verifies rollback after a simulated crash, tests a phantom-row/TOCTOU case, measures latency, and checks the monotonic safe-core property over 10,000 randomized trials.

In [ ]:
# CELL 1 — Imports and configuration
from pathlib import Path
import os,json,sqlite3,hmac,hashlib,random,time,uuid,math,zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED=20261006; N_PER_CONDITION=120
CONDITIONS=['VALID','ARG_TAMPER','FORGED_SIGNATURE','POLICY_CHANGE','TARGET_TAMPER','REPLAY','STALE_SNAPSHOT','EXPIRED','ACTOR_MISMATCH','TOOL_MISMATCH','CRASH_BEFORE_COMMIT']
RUN_ID=time.strftime('%Y%m%d_%H%M%S'); RUN_DIR=(Path('/content') if Path('/content').exists() else Path.cwd())/f'NTX_BROKER_{RUN_ID}'; RUN_DIR.mkdir(parents=True,exist_ok=True)
def sha256_file(path):
    h=hashlib.sha256();
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def atomic_json(obj,path):
    p=Path(path); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2,sort_keys=True)); tmp.replace(p)
print('RUN_DIR =',RUN_DIR)

In [ ]:
# CELL 2 — Transaction-first verified broker

def canonical_json(x):return json.dumps(x,sort_keys=True,separators=(',',':'),ensure_ascii=False)
def sign(payload,secret):return hmac.new(secret,canonical_json(payload).encode(),hashlib.sha256).hexdigest()
class Broker:
    def __init__(self,db_path,secret):self.db_path=str(db_path);self.secret=secret;self.init_db()
    def con(self):
        c=sqlite3.connect(self.db_path,timeout=30,isolation_level=None,check_same_thread=False);c.row_factory=sqlite3.Row;return c
    def init_db(self):
        c=self.con();c.execute('PRAGMA journal_mode=WAL');c.execute('CREATE TABLE IF NOT EXISTS nonces(nonce TEXT PRIMARY KEY, committed_at REAL)');c.execute('CREATE TABLE IF NOT EXISTS objects(id INTEGER PRIMARY KEY, owner TEXT, status TEXT, version INTEGER NOT NULL DEFAULT 0)');c.close()
    def reset(self,objects):
        c=self.con();c.execute('BEGIN IMMEDIATE');c.execute('DELETE FROM nonces');c.execute('DELETE FROM objects')
        for o in objects:c.execute('INSERT INTO objects(id,owner,status,version) VALUES (?,?,?,?)',(o['id'],o['owner'],o['status'],o.get('version',0)))
        c.execute('COMMIT');c.close()
    def snapshot(self,owner):
        c=self.con();rows=c.execute('SELECT id,version,status FROM objects WHERE owner=? ORDER BY id',(owner,)).fetchall();c.close();return [{'id':int(r['id']),'version':int(r['version']),'status':r['status']} for r in rows]
    def issue(self,actor,tool,args,predicted_ids,ttl_s=120):
        now=time.time();payload={'nonce':uuid.uuid4().hex,'actor':actor,'tool':tool,'args':args,'predicted_ids':sorted(map(int,predicted_ids)),'snapshot':self.snapshot(args['owner']),'issued_at':now,'expires_at':now+ttl_s};return {'payload':payload,'signature':sign(payload,self.secret)}
    def verify(self,cap):return hmac.compare_digest(sign(cap['payload'],self.secret),str(cap.get('signature','')))
    def execute(self,cap,actor,policy_fn,simulate_crash=False):
        t0=time.perf_counter()
        if not self.verify(cap):return {'ok':False,'reason':'BAD_SIGNATURE','latency_ms':(time.perf_counter()-t0)*1000}
        p=cap['payload']
        if time.time()>p['expires_at']:return {'ok':False,'reason':'EXPIRED','latency_ms':(time.perf_counter()-t0)*1000}
        if p['actor']!=actor:return {'ok':False,'reason':'ACTOR_MISMATCH','latency_ms':(time.perf_counter()-t0)*1000}
        if p['tool']!='archive_objects':return {'ok':False,'reason':'TOOL_MISMATCH','latency_ms':(time.perf_counter()-t0)*1000}
        c=self.con()
        try:
            c.execute('BEGIN IMMEDIATE')
            if c.execute('SELECT 1 FROM nonces WHERE nonce=?',(p['nonce'],)).fetchone():c.execute('ROLLBACK');return {'ok':False,'reason':'REPLAY','latency_ms':(time.perf_counter()-t0)*1000}
            if not policy_fn(actor,p['args']):c.execute('ROLLBACK');return {'ok':False,'reason':'POLICY_CHANGED','latency_ms':(time.perf_counter()-t0)*1000}
            rows=c.execute('SELECT id,version,status FROM objects WHERE owner=? ORDER BY id',(p['args']['owner'],)).fetchall();current=[{'id':int(r['id']),'version':int(r['version']),'status':r['status']} for r in rows]
            if current!=p['snapshot']:c.execute('ROLLBACK');return {'ok':False,'reason':'STALE_SNAPSHOT','latency_ms':(time.perf_counter()-t0)*1000}
            current_ids=[r['id'] for r in current if r['status']=='OPEN']
            if current_ids!=p['predicted_ids']:c.execute('ROLLBACK');return {'ok':False,'reason':'TARGET_SET_CHANGED','latency_ms':(time.perf_counter()-t0)*1000}
            c.execute('INSERT INTO nonces(nonce,committed_at) VALUES (?,?)',(p['nonce'],time.time()))
            for oid in current_ids:c.execute("UPDATE objects SET status='ARCHIVED',version=version+1 WHERE id=?",(oid,))
            if simulate_crash:raise RuntimeError('SIMULATED_CRASH_BEFORE_COMMIT')
            c.execute('COMMIT');return {'ok':True,'reason':'COMMITTED','ids':current_ids,'latency_ms':(time.perf_counter()-t0)*1000}
        except RuntimeError as e:
            try:c.execute('ROLLBACK')
            except Exception:pass
            if str(e)=='SIMULATED_CRASH_BEFORE_COMMIT':return {'ok':False,'reason':'CRASH_ROLLBACK','latency_ms':(time.perf_counter()-t0)*1000}
            raise
        finally:c.close()

In [ ]:
# CELL 3 — Run 1,320 randomized scenarios
rng=random.Random(SEED);db=RUN_DIR/'broker.sqlite';secret=hashlib.sha256(f'secret-{SEED}'.encode()).digest();broker=Broker(db,secret)
def make_objects(s):
    rr=random.Random(s);objs=[];oid=1
    for _ in range(rr.randint(1,6)):objs.append({'id':oid,'owner':'alice','status':'OPEN','version':0});oid+=1
    for _ in range(rr.randint(1,4)):objs.append({'id':oid,'owner':'bob','status':'OPEN','version':0});oid+=1
    return objs
allow=lambda actor,args:actor=='alice' and args.get('owner')=='alice';deny=lambda actor,args:False
records=[]
for ci,cond in enumerate(CONDITIONS):
    for j in range(N_PER_CONDITION):
        objects=make_objects(SEED+ci*10000+j);broker.reset(objects);pred=[o['id'] for o in objects if o['owner']=='alice'];cap=broker.issue('alice','archive_objects',{'owner':'alice'},pred,ttl_s=60);actor='alice';policy=allow;crash=False
        if cond=='ARG_TAMPER':cap['payload']['args']['owner']='bob'
        elif cond=='FORGED_SIGNATURE':cap['signature']='0'*64
        elif cond=='POLICY_CHANGE':policy=deny
        elif cond=='TARGET_TAMPER':cap['payload']['predicted_ids']=pred[:-1] if len(pred)>1 else pred+[999];cap['signature']=sign(cap['payload'],secret)
        elif cond=='REPLAY':
            broker.execute(cap,'alice',allow);out=broker.execute(cap,'alice',allow);records.append({'condition':cond,'trial':j,'ok':out['ok'],'reason':out['reason'],'expected_ok':False,'latency_ms':out['latency_ms']});continue
        elif cond=='STALE_SNAPSHOT':
            c=broker.con();c.execute('BEGIN IMMEDIATE');c.execute('UPDATE objects SET version=version+1 WHERE id=?',(pred[0],));c.execute('COMMIT');c.close()
        elif cond=='EXPIRED':cap['payload']['expires_at']=time.time()-1;cap['signature']=sign(cap['payload'],secret)
        elif cond=='ACTOR_MISMATCH':actor='mallory'
        elif cond=='TOOL_MISMATCH':cap['payload']['tool']='delete_objects';cap['signature']=sign(cap['payload'],secret)
        elif cond=='CRASH_BEFORE_COMMIT':crash=True
        out=broker.execute(cap,actor,policy,simulate_crash=crash);records.append({'condition':cond,'trial':j,'ok':out['ok'],'reason':out['reason'],'expected_ok':cond=='VALID','latency_ms':out['latency_ms']})
df=pd.DataFrame(records);assert len(df)==1320;df['correct']=df['ok']==df['expected_ok'];df.to_csv(RUN_DIR/'broker_1320_results.csv',index=False);display(df.groupby('condition').agg(n=('correct','size'),correct=('correct','mean'),committed=('ok','mean'),median_ms=('latency_ms','median')));print('Overall correct behavior:',df['correct'].mean())

In [ ]:
# CELL 4 — Crash rollback + phantom-row/TOCTOU checks
objects=make_objects(SEED+999999);broker.reset(objects);pred=[o['id'] for o in objects if o['owner']=='alice'];cap=broker.issue('alice','archive_objects',{'owner':'alice'},pred,ttl_s=60);before=broker.snapshot('alice');crash_out=broker.execute(cap,'alice',allow,simulate_crash=True);after=broker.snapshot('alice');assert before==after and crash_out['reason']=='CRASH_ROLLBACK';print('Crash rollback: PASS')
objects=make_objects(SEED+888888);broker.reset(objects);pred=[o['id'] for o in objects if o['owner']=='alice'];cap=broker.issue('alice','archive_objects',{'owner':'alice'},pred,ttl_s=60);c=broker.con();c.execute('BEGIN IMMEDIATE');new_id=max(o['id'] for o in objects)+100;c.execute("INSERT INTO objects(id,owner,status,version) VALUES (?,?,?,0)",(new_id,'alice','OPEN'));c.execute('COMMIT');c.close();phantom=broker.execute(cap,'alice',allow);print('Phantom result:',phantom);assert not phantom['ok'] and phantom['reason'] in {'STALE_SNAPSHOT','TARGET_SET_CHANGED'};print('Phantom-row check: PASS')

In [ ]:
# CELL 5 — Empirical monotonic safe-core property

def safe_core(sets):
    if not sets:return set()
    k=set(sets[0])
    for s in sets[1:]:k &= set(s)
    return k
rng=np.random.default_rng(SEED);violations=0;N_PROPERTY=10000;universe=list(range(20))
for _ in range(N_PROPERTY):
    sets=[]
    for __ in range(int(rng.integers(1,6))):sets.append({u for u,keep in zip(universe,rng.random(len(universe))<.5) if keep})
    k1=safe_core(sets);extra={u for u,keep in zip(universe,rng.random(len(universe))<.5) if keep};k2=safe_core(sets+[extra]);violations+=int(not k2.issubset(k1))
print('Monotonicity trials:',N_PROPERTY,'violations:',violations);assert violations==0

In [ ]:
# CELL 6 — Latency summary and export
summary=df.groupby('condition').agg(n=('correct','size'),correct_rate=('correct','mean'),commit_rate=('ok','mean'),median_latency_ms=('latency_ms','median'),p95_latency_ms=('latency_ms',lambda x:float(x.quantile(.95))),p99_latency_ms=('latency_ms',lambda x:float(x.quantile(.99)))).reset_index();summary.to_csv(RUN_DIR/'broker_summary.csv',index=False);display(summary)
fig,ax=plt.subplots(figsize=(11,4.8));x=np.arange(len(summary));ax.bar(x,summary['median_latency_ms']);ax.set_xticks(x);ax.set_xticklabels(summary['condition'],rotation=50,ha='right');ax.set_ylabel('Median latency (ms)');ax.set_title('Verified broker latency');fig.tight_layout();fig.savefig(RUN_DIR/'broker_latency.png',dpi=300,bbox_inches='tight');plt.show()
overall={'n_scenarios':len(df),'behavior_correct_rate':float(df['correct'].mean()),'valid_commit_rate':float(df.loc[df['condition']=='VALID','ok'].mean()),'invalid_fail_closed_rate':float((~df.loc[df['condition']!='VALID','ok']).mean()),'phantom_row_fail_closed':bool(not phantom['ok']),'crash_rollback_passed':bool(before==after),'monotonicity_trials':N_PROPERTY,'monotonicity_violations':violations};atomic_json(overall,RUN_DIR/'overall_results.json');print(json.dumps(overall,indent=2));assert overall['behavior_correct_rate']==1.0 and overall['valid_commit_rate']==1.0 and overall['invalid_fail_closed_rate']==1.0 and overall['phantom_row_fail_closed'] and overall['crash_rollback_passed'] and overall['monotonicity_violations']==0
zip_path=RUN_DIR.with_suffix('.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(RUN_DIR))
print('RESULT ZIP:',zip_path,'SHA256:',sha256_file(zip_path))
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:pass